# Case study: which constructed features help a Berlin price model?

*Session 9, block 1 practice. Author: course team, licence CC-BY-4.0.*

**Question.** What drives the nightly price of a Berlin Airbnb listing, and which features built from dates, location, lists and categories improve a price model? We build the features of [theory page 1](../theory/01-dates-interactions-and-categories.md) and measure the gain of each group with host-grouped cross-validation (Session 7), for a linear model and for gradient boosting.

**Data.** Inside Airbnb, Berlin, snapshot of 26 June 2026 (CC BY 4.0), prepared with `uv run python case-study/prepare_airbnb.py`. Collected from public listing pages; the course copy contains no host names. Use `host_id` only as a grouping key and report in aggregate. The notebook runs in about two minutes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
AIRBNB = ROOT / "case-study" / "data" / "airbnb"
import json

import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler, TargetEncoder

listings = pd.read_parquet(AIRBNB / "listings.parquet")
# short-stay listings with a price (minimum stay < 28 nights, as in Sessions 4-6),
# without the 26 outliers priced below EUR 10 or above EUR 1,000
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,      # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)
print(len(listings), "listings in the snapshot,", len(bnb), "in the price-model table,",
      bnb["host_id"].nunique(), "hosts")
y = np.log(bnb["price"])
hosts = bnb["host_id"]
cv = GroupKFold(5)

## 1. Dates

Elapsed-time features for each listing, measured at the snapshot. Do they say anything about the price?

In [ ]:
dates = pd.DataFrame({
    "host_years": bnb["hosts_time_as_host_years"],
    "listing_age_days": (bnb["last_scraped"] - bnb["first_review"]).dt.days,
    "days_since_last_review": (bnb["last_scraped"] - bnb["last_review"]).dt.days,
})
print(dates.isna().mean().round(3).to_dict())
print(dates.corrwith(y, method="spearman").round(3).to_dict())
bnb.groupby(pd.cut(dates["host_years"], [-1, 1, 3, 6, 10, 20]), observed=True)["price"].agg(["count", "median"])

Dates matter for **demand** rather than price. Reviews per calendar month (all Berlin listings in the snapshot), and the cyclical encoding of the month:

In [ ]:
reviews = pd.read_parquet(AIRBNB / "reviews_monthly.parquet")
per_month = reviews.groupby("month")["n_reviews"].sum()
fig, ax = plt.subplots(figsize=(10, 3.5))
per_month["2016-01-01":"2026-06-01"].plot(ax=ax)
ax.set_ylabel("reviews per month")
ax.set_title("Reviews of Berlin listings per month (proxy for stays)")
plt.show()
recent = per_month["2023-01-01":"2025-12-01"]
season = pd.DataFrame({"share": recent.groupby(recent.index.month).sum() / recent.sum()})
season["month_sin"] = np.sin(2 * np.pi * season.index / 12)
season["month_cos"] = np.cos(2 * np.pi * season.index / 12)
season.round(3)

**Task 1.** Which months are busiest? Why would a model of monthly demand need `month_sin` and `month_cos` (or month dummies), while a price model built on one snapshot has no use for the month?

## 2. Location: distance to the centre

Latitude and longitude are two numbers whose relation to price is not linear. One constructed number, the distance to Alexanderplatz, captures the main pattern.

In [ ]:
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2, (bnb["longitude"] - 13.4132) * 68.0)
bins = pd.cut(bnb["dist_km"], [0, 2, 4, 6, 10, 30])
bnb.pivot_table(index=bins, columns="room_type", values="price", aggfunc="median", observed=True).round(0)

## 3. An interaction: room type × guests

On the log scale `log(accommodates)` has an elasticity as coefficient. Does it differ between entire flats and private rooms?

In [ ]:
base = pd.get_dummies(bnb["room_type"], dtype=int).drop(columns="Private room")
base["log_guests"] = np.log(bnb["accommodates"])
base["dist_km"] = bnb["dist_km"]
with_inter = base.assign(log_guests_x_entire=base["log_guests"] * base["Entire home/apt"])
for target_name, target in [("log price", y), ("price in EUR", bnb["price"])]:
    for name, Xi in [("additive", base), ("+ interaction", with_inter)]:
        r2 = cross_val_score(LinearRegression(), Xi, target, cv=cv, groups=hosts, scoring="r2").mean()
        print(f"{target_name:13s} {name:14s} R² {r2:.3f}")
print(dict(zip(with_inter.columns, LinearRegression().fit(with_inter, y).coef_.round(3))))

**Task 3.** Translate the two elasticities into "doubling the guests raises the price by … %" for rooms and for flats. Why does the interaction help more on the euro scale than on the log scale? Try one more interaction of your own (for example distance × entire flat).

## 4. Rare categories: property types

In [ ]:
counts = bnb["property_type"].value_counts()
print(len(counts), "property types;", (counts < 20).sum(), "with fewer than 20 listings")
print(counts.tail(8).to_dict())
for k in [1, 5, 20, 100]:
    enc = OneHotEncoder(min_frequency=k, handle_unknown="infrequent_if_exist")
    print(k, enc.fit(bnb[["property_type"]]).transform(bnb[["property_type"]]).shape[1], "columns")

## 5. Target encoding of the neighbourhood, and the leak

We hold out 30 % of the hosts. A naive encoding uses each row's own price; `TargetEncoder` cross-fits.

In [ ]:
tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=0).split(bnb, groups=hosts))
r2_col = lambda target, feature: round(np.corrcoef(target, feature)[0, 1] ** 2, 3)
rows = []
for col in ["neighbourhood", "property_type", "host_id"]:
    naive = y.iloc[tr].groupby(bnb[col].iloc[tr]).mean()
    f_tr = bnb[col].iloc[tr].map(naive)
    f_te = bnb[col].iloc[te].map(naive).fillna(y.iloc[tr].mean())
    enc = TargetEncoder(target_type="continuous", cv=KFold(5, shuffle=True, random_state=0))
    e_tr = enc.fit_transform(bnb[[col]].iloc[tr], y.iloc[tr]).ravel()
    e_te = enc.transform(bnb[[col]].iloc[te]).ravel()
    rows.append({"column": col, "levels": bnb[col].nunique(),
                 "naive train R²": r2_col(y.iloc[tr], f_tr), "naive new-host R²": r2_col(y.iloc[te], f_te),
                 "cross-fitted train R²": r2_col(y.iloc[tr], e_tr), "cross-fitted new-host R²": r2_col(y.iloc[te], e_te)})
pd.DataFrame(rows)

**Task 5.** Explain each row in one sentence. Compute the smoothed encoding by hand for one small neighbourhood with *m* = 10 and compare it with the naive mean.

## 6. All features in one pipeline, and which groups earn their place

Custom transformers from theory page 1: the distance (`FunctionTransformer`) and the amenity indicators (a class that learns the amenity list in `fit`).

In [ ]:
def km_to_centre(X):
    return pd.DataFrame({"dist_km": np.hypot((X["latitude"] - 52.5219) * 111.2,
                                             (X["longitude"] - 13.4132) * 68.0)}, index=X.index)


def review_dates(X):
    """Days since first and last review at the snapshot; missing for listings without reviews."""
    return pd.DataFrame({"listing_age_days": (X["last_scraped"] - X["first_review"]).dt.days,
                         "days_since_last_review": (X["last_scraped"] - X["last_review"]).dt.days}, index=X.index)


class AmenityIndicators(BaseEstimator, TransformerMixin):
    """One 0/1 column per amenity listed by at least min_count training listings."""

    def __init__(self, min_count=50):
        self.min_count = min_count

    def fit(self, X, y=None):
        counts = X.iloc[:, 0].map(json.loads).explode().value_counts()
        self.amenities_ = sorted(counts[counts >= self.min_count].index)
        return self

    def transform(self, X):
        lists = X.iloc[:, 0].map(lambda s: set(json.loads(s)))
        return pd.DataFrame({f"am_{a}": lists.map(lambda x, a=a: a in x).astype(int) for a in self.amenities_},
                            index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"am_{a}" for a in self.amenities_], dtype=object)


groups = {
    "size + room type": [("size", "passthrough", ["accommodates", "bedrooms", "beds", "bathrooms"]),
                         ("room", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["room_type"])],
    "+ distance": [("dist", FunctionTransformer(km_to_centre, feature_names_out=lambda tf, n: ["dist_km"]),
                    ["latitude", "longitude"])],
    "+ district": [("district", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["district"])],
    "+ neighbourhood (target-encoded)": [("nbhd", TargetEncoder(target_type="continuous",
                                                                cv=KFold(5, shuffle=True, random_state=0)),
                                          ["neighbourhood"])],
    "+ property type (rare grouped)": [("ptype", OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist",
                                                               sparse_output=False), ["property_type"])],
    "+ amenities": [("amen", AmenityIndicators(min_count=50), ["amenities"])],
    "+ reviews and review dates": [("rev", "passthrough", ["number_of_reviews", "review_scores_rating"]),
                                   ("revdates", FunctionTransformer(review_dates, feature_names_out=lambda tf, n:
                                                                    ["listing_age_days", "days_since_last_review"]),
                                    ["last_scraped", "first_review", "last_review"])],
}
X = bnb.drop(columns=["price"])


def evaluate(steps, model):
    pipe = make_pipeline(ColumnTransformer(steps), model)
    pred = cross_val_predict(pipe, X, y, cv=cv, groups=hosts, n_jobs=-1)
    return r2_score(y, pred), mean_absolute_error(bnb["price"], np.exp(pred))


linear = lambda: make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), Ridge(alpha=10))
rows, steps = [], []
for name, new in groups.items():
    steps = steps + new
    r2_lin, mae_lin = evaluate(steps, linear())
    r2_gbm, mae_gbm = evaluate(steps, HistGradientBoostingRegressor(random_state=0))
    rows.append({"features": name, "ridge R²": r2_lin, "ridge MAE €": mae_lin,
                 "boosting R²": r2_gbm, "boosting MAE €": mae_gbm})
pd.DataFrame(rows).round(3)

**Task 6.** (1) Which feature group brings the largest gain for each model? Which groups bring nothing or make a model worse? (2) Why might the target-encoded neighbourhood help the linear model more than the boosted one? (3) The review-date features are fine for describing existing listings. Would you keep them in a price tool for **new** hosts? (4) Try `AmenityIndicators(min_count=10)` and `min_count=500`.

**Team project.** Document your project's feature set: for each feature, its source, the moment at which it is known, and the pipeline step that computes it.